# DevAPI01.ipynb: Developing `estatpy.api.get_csv_data()`

This module is inspired by [cocosan](https://www.youtube.com/watch?v=hiaK-jTXpCI)'s video example <cite data-cite="cocosan2023python.en">(Cocosan, 2023)</cite>. Cocosan uses Python packages streamlit, pandas, and plotly. This notebook will first partially implement Cocosan's example to retrieve a CVS stream from e-Stat and create a Python data frame. Knowledge gained from this exercise will be applied to create a first version of an api.getdata() function.

## Check for required packages

Cocosan uses streamlit to produce output in markdown that is then displayed in a browser. This implementation displays the output in this notebook. The following code runs terminal commands to install the most recent versions of pandas, and plotly.

In [1]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install plotly 
%pip install python-dotenv # for using local environment variables
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install requests

## Obtain and save API appId

E-Stat's API requires registration at [e-Stat API](https://www.e-stat.go.jp/api/en). Registration and sign-in accesses a dashboard from which you can apply for an App ID. 

When stored in a text file (.env) `ESTAT_APP_ID = '(App Id)'` in your project root, it can be retrieved with the following code.[^1]

[^1] Cocosan uses the [st.secrets feature of streamlit](https://docs.streamlit.io/develop/api-reference/connections/st.secrets).

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
app_id = os.environ['ESTAT_APP_ID']
# print(app_id)

## Define the `get_data(url)` function

Now define the `get_data(url)` function. This implementation uses `url.split` to divide the url at the field string `appId=`. `url.split` returns a vector of string segments and, because "appId=" only occurs once, the returned vector here is of length 2. These are concatenated with "appId=" and the variable `appId` to produce the complete url.

Cocosan provides space for up to 50 columns (`names=range(50)`) and has [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) read the file to obtain a [pandas.DataFrame](https://pandas.pydata.org/docs/reference/frame.html).

The data frame `df` begins with several rows of metadata terminated by the string "VALUE". 

Cocosan' example uses an API url that downloads data on [monthly housing starts](https://www.e-stat.go.jp/stat-search/database?page=1&layout=datalist&toukei=00600120&tstat=000001016966&cycle=1&statdisp_id=0003114535&tclass1val=0). This table is in Japanese. Housing start data is available in English but only in the form of [monthly Excel tables](https://www.e-stat.go.jp/en/stat-search/files?page=1&layout=dataset&toukei=00600120&tstat=000001016966&cycle=1&tclass1val=0).

A smaller English dataset is the [Labour Force Survey Basic Tabulation Whole Japan Monthly table Population of 15 years old and over by labour force status](https://www.e-stat.go.jp/en/dbview?sid=0003005798). The API url for that table is assigned to `enurl` below.

In [3]:
# Cocosan's url:
cocosanurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=J&statsDataId=0003114535&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'
# Labor force url:
enurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'


Technical references consulted for the code below:

   * [Iterating through files chunk by chunk](https://pandas.pydata.org/docs/user_guide/io.html#iterating-through-files-chunk-by-chunk)
   * [Python While Loops](https://www.w3schools.com/python/python_while_loops.asp)
   * [Reading and Writing Files](https://docs.python.org/3/tutorial/inputoutput.html#tut-files)
   * [readers.py](https://github.com/pandas-dev/pandas/blob/main/pandas/io/parsers/readers.py)
   * [Read a file line by line in Python](https://www.geeksforgeeks.org/python/read-a-file-line-by-line-in-python/)
   * [io — Core tools for working with streams](https://docs.python.org/3/library/io.html#text-i-o)
   * [StreamReader Objects](https://docs.python.org/3/library/codecs.html#streamreader-objects)
   * [Requests: HTTP for Humans™](https://requests.readthedocs.io/en/latest/)
      * [Streaming Requests](https://requests.readthedocs.io/en/latest/user/advanced/#streaming-requests)
   * [tempfile — Generate temporary files and directories](https://docs.python.org/3.13/library/tempfile.html)



## Work-around for metadata in csv stream

CSV streams from the e-Stat API have a multiple line preface of metadata that describes the dataset and that may be important for documenting the study. A problem arises because `pandas.get_csv` takes its column count from the first line, and this cannot be changed midstream even with chunking. The work-around below iterates over the `response.iter_lines()` stream from `requests.get()` to move the stream position to the start of the table, writing the metadata to a temporary file in the process. (Setting `stream` to `False` prevents `response.iter_lines()` from chunking the stream and being rendered reentrant unsafe.) Taking a hint from [pandas.parsers._read](https://github.com/pandas-dev/pandas/blob/main/pandas/io/parsers/readers.py#258), it turns out that `pandas.read_csv` can be launched here and will grab the next row as its column names.

This implementation includes a `description` parameter that gives the user a mechanism for writing documentation with the dataset. The default is 'datetime.datetime.now()', but this can be substituted by the user as she may desire.

In [10]:
import pandas as pd
import os
import requests
import tempfile
import re
import datetime
def get_data(url, description = datetime.datetime.now()):
    app_id = os.environ['ESTAT_APP_ID']
    url_split = url.split("appId=")
    url = url_split[0] + "appId=" + app_id + url_split[1]

    # the csv has several rows of metadata terminated by a row starting with "VALUE".
    # The data table starts on the next row.
    # Put the metadata in a StringIO
    result = {}
    with requests.get(url,stream=False) as estatresponse: # chunking in iter_lines doesn't work for stream=True
        if estatresponse.encoding is None:
            estatresponse.encoding = 'utf-8'
        estatlines = estatresponse.iter_lines(chunk_size=1024, decode_unicode=True)
        with tempfile.NamedTemporaryFile(mode='w',delete_on_close=False,encoding = 'utf-8') as fheader:
            with tempfile.NamedTemporaryFile(mode='w',delete_on_close=False,encoding = 'utf-8') as fp:
                inheader = True
                colnum = 0
                for line in estatlines:
                    if inheader == True:
                        #count columns
                        fields = re.split('","',line)
                        if len(fields) > colnum :
                            colnum = len(fields)
                        fheader.write(line)
                        fheader.write("\n")
                        if( line.startswith('"VALUE"')):
                            inheader = False
                            fheader.flush()
                            fheader.seek(0)
                    else:
                        fp.write(line)
                        fp.write("\n")
                fheader.close()
                fp.close()
                dfHeader = pd.read_csv(fheader.name, names = range(colnum))
                dfHeader = dfHeader.dropna(axis=1, how = "all")
                dfMain = pd.read_csv(fp.name)
                result['Description'] = description
                result['Header'] = dfHeader
                result['Main'] = dfMain
    return result

        
dfs = get_data(enurl,description=datetime.datetime.now())
print(dfs.get('Description'))
print(dfs.get('Header'))

2026-03-02 14:20:26.686850
                       0                                                  1  \
0                 RESULT                                                NaN   
1                 STATUS                                                  0   
2              ERROR_MSG       The process has been successfully completed.   
3                   DATE                      2026-03-02T14:20:27.122+09:00   
4             RESULT_INF                                                NaN   
5           TOTAL_NUMBER                                               4680   
6            FROM_NUMBER                                                  1   
7              TO_NUMBER                                               4680   
8              TABLE_INF                                         0003005798   
9              STAT_NAME                                           00200531   
10               GOV_ORG                                              00200   
11       STATISTICS_NAME 

In [6]:
print(dfs.get('Main'))

      tab_code Tabulated variable  cat01_code        Industry  cat02_code  \
0            1      Actual number           0  All industries           0   
1            1      Actual number           0  All industries           0   
2            1      Actual number           0  All industries           0   
3            1      Actual number           0  All industries           0   
4            1      Actual number           0  All industries           0   
...        ...                ...         ...             ...         ...   
4675         1      Actual number           0  All industries           9   
4676         1      Actual number           0  All industries           9   
4677         1      Actual number           0  All industries           9   
4678         1      Actual number           0  All industries           9   
4679         1      Actual number           0  All industries           9   

                     Labour force status  cat03_code         Sex  area_code

## Observations

* The metadata should be kept with the table. The pandas DataFrame has an experimental [attributes member](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.attrs.html), but it doesn't appear to be suitable and may change without warning.

* One option is to create a class with the metadata as a `dict` and table as a `pandas.DataFrame`. See [class type assignability](https://typing.python.org/en/latest/spec/class-compat.html#classvar).

## References

.. bibliography::
    :filter: docname in docnames